Author: **Dongyuan Gao** **Solene**

Course: HSLU Computer Vision — Lecture 3 Project

Based on the style of the lecturer's notebooks by *Safouane El Ghazouali* (TOELT LLC / HSLU).

# -----  -----  -----  -----  -----  -----  -----  -----

# Fine-Tuning YOLO26 for Self-Driving Object Detection

In this notebook we fine-tune a pre-trained **YOLO26s** on the **Udacity Self-Driving Car dataset** (≈15,000 dashcam images, 11 classes: car, truck, pedestrian, biker, traffic light, traffic sign, …).

The goal: build a **real-time detector** that finds vehicles, pedestrians and traffic signals in dashcam footage — the core perception step of any self-driving stack.

### Why YOLO?
- **Real-time**: a single forward pass gives all boxes + labels.
- **Detection, not classification**: tells us *what* AND *where*.
- **Easy fine-tuning** via the Ultralytics library.

### What You'll Learn
- Downloading a pre-labelled detection dataset from Roboflow.
- Fine-tuning YOLO26 on custom classes.
- Reading training curves and validation metrics (mAP).
- Running inference on images and videos.
- Plugging the fine-tuned weights into a **live webcam** demo on your Mac.

#  Running on DGX via VS Code Remote

Project directory on DGX: `/home/dongyuan/Desktop/computer_vision`

Typical flow:
- Connect to the DGX with VS Code Remote - SSH.
- Open this notebook **on the remote machine** (so paths refer to DGX storage).
- Use a conda env or venv with PyTorch + CUDA already installed.
- Keep datasets on DGX local storage (faster than network mounts).

#  Environment Setup (DGX)

Install Ultralytics (YOLO), Roboflow (dataset download), and OpenCV.

On a DGX, you typically already have a CUDA-enabled PyTorch in your conda env.
If you do not, create or activate your environment before running the install below.

In [ ]:
!pip install -q ultralytics roboflow opencv-python

### Optional: Ollama Python Client (local VLM captions)

If you want to run the VLM overlay cell later, install the **Python client** in your environment.
The Ollama server itself is installed and run in the terminal (system-level).

Example install (terminal or notebook cell): `pip install ollama`

### Import Libraries & Check GPU

On the DGX you should see `cuda` and at least one visible GPU.
If it prints `cpu`, your environment is missing CUDA-enabled PyTorch or no GPU is visible.

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os, glob, yaml
import cv2
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# Quick GPU visibility check on DGX
!nvidia-smi -L

# Explanation
# - device: tells YOLO where to run (GPU is ~30x faster than CPU).
# - Ultralytics auto-uses this device unless we override it.

# Dataset on the DGX (Roboflow or Local Path)

You can either download with Roboflow **on the DGX** or point to a dataset that is already on DGX storage.

**Option A (Roboflow download on DGX):**
1. Go to https://public.roboflow.com/object-detection/self-driving-car
2. Click **Download Dataset** → pick **YOLOv8** format (compatible with v10).
3. Roboflow shows you a **personalized snippet** with your API key — paste it in the next cell.

**Option B (Dataset already on DGX):**
- Set the `DATASET_DIR` path below to the folder that contains `data.yaml`, `train/`, `valid/`, `test/`.

**Note (local path):** If you set `USE_ROBOFLOW = False`, this notebook looks for the dataset in `./Self-Driving-Car-3` or `./self-driving-car`. You can also override with an environment variable, e.g. `export DATASET_DIR=/path/to/dataset`.


In [ ]:
# Set this to False if the dataset is already on DGX storage
USE_ROBOFLOW = False

# If USE_ROBOFLOW is False, set the local dataset folder on DGX
def resolve_dataset_dir() -> str:
    env_path = os.getenv("DATASET_DIR")
    if env_path:
        return env_path
    candidates = [
        os.path.join(os.getcwd(), "Self-Driving-Car-3"),
        os.path.join(os.getcwd(), "self-driving-car"),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    raise FileNotFoundError(
        "Dataset folder not found. Set DATASET_DIR or place dataset at ./Self-Driving-Car-3 or ./self-driving-car"
    )

if USE_ROBOFLOW:
    # ---- PASTE YOUR ROBOFLOW SNIPPET HERE ----
    rf = Roboflow(api_key="YOUR_API_KEY")
    project = rf.workspace("roboflow-gw7yv").project("self-driving-car")
    dataset = project.version(3).download("yolov8")
    dataset_location = dataset.location
else:
    DATASET_DIR = resolve_dataset_dir()
    dataset_location = DATASET_DIR

data_yaml = os.path.join(dataset_location, "data.yaml")
print(f"Dataset location: {dataset_location}")
print(f"data.yaml: {data_yaml}")

# Explanation
# - dataset_location: absolute path to the dataset folder on DGX
# - data.yaml lists class names and the train/valid/test paths YOLO needs

In [ ]:
# If Roboflow export-only layout, create train/valid/test splits
from pathlib import Path
import random
import shutil

export_images = Path(dataset_location) / "export" / "images"
export_labels = Path(dataset_location) / "export" / "labels"

if export_images.is_dir() and export_labels.is_dir():
    split_root = Path(dataset_location)
    split_names = {"train": 0.8, "valid": 0.1, "test": 0.1}
    for split in split_names:
        (split_root / split / "images").mkdir(parents=True, exist_ok=True)
        (split_root / split / "labels").mkdir(parents=True, exist_ok=True)

    image_files = sorted(export_images.glob("*.jpg"))
    if not image_files:
        image_files = sorted(export_images.glob("*.png"))
    if not image_files:
        raise FileNotFoundError(f"No images found in {export_images}")

    random.seed(42)
    random.shuffle(image_files)
    n_total = len(image_files)
    n_train = int(n_total * split_names["train"])
    n_valid = int(n_total * split_names["valid"])
    train_files = image_files[:n_train]
    valid_files = image_files[n_train : n_train + n_valid]
    test_files = image_files[n_train + n_valid :]

    def copy_pair(img_path: Path, split: str) -> None:
        lbl_path = export_labels / (img_path.stem + ".txt")
        shutil.copy2(img_path, split_root / split / "images" / img_path.name)
        if lbl_path.exists():
            shutil.copy2(lbl_path, split_root / split / "labels" / lbl_path.name)

    for p in train_files:
        copy_pair(p, "train")
    for p in valid_files:
        copy_pair(p, "valid")
    for p in test_files:
        copy_pair(p, "test")

    data_yaml = str(split_root / "data.yaml")
    print(f"Created splits under: {split_root}")
    print(f"Using data.yaml: {data_yaml}")
else:
    print("Export folder not found; using existing train/valid/test splits.")

### Inspect the Dataset

Before training, always look at a few images to confirm labels make sense.
YOLO labels are **normalised** (values between 0 and 1) in the format:

````
class  x_center  y_center  width  height
````

We convert them back to pixel coordinates to draw the boxes.

In [ ]:
from pathlib import Path

if 'dataset_location' not in globals() or not dataset_location:
    env_path = os.getenv('DATASET_DIR')
    if env_path and os.path.isdir(env_path):
        dataset_location = env_path
    else:
        candidates = [
            os.path.join(os.getcwd(), 'Self-Driving-Car-3'),
            os.path.join(os.getcwd(), 'self-driving-car'),
        ]
        dataset_location = next((path for path in candidates if os.path.isdir(path)), '')
    if not dataset_location:
        raise FileNotFoundError(
            'Dataset folder not found. Run the dataset setup cell first, set DATASET_DIR, or place the dataset at ./Self-Driving-Car-3 or ./self-driving-car.'
        )

if 'data_yaml' not in globals() or not data_yaml:
    data_yaml = os.path.join(dataset_location, 'data.yaml')

if not os.path.isfile(data_yaml):
    raise FileNotFoundError(f'data.yaml not found: {data_yaml}')

images_dir = f"{dataset_location}/train/images"
labels_dir = f"{dataset_location}/train/labels"

# Fallback for Roboflow export-only layout
export_images = os.path.join(dataset_location, 'export', 'images')
export_labels = os.path.join(dataset_location, 'export', 'labels')
if not os.path.isdir(images_dir) and os.path.isdir(export_images):
    images_dir = export_images
    labels_dir = export_labels

# Read class names from data.yaml
with open(data_yaml) as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg['names']
print(f'Using dataset: {dataset_location}')
print(f'Using data.yaml: {data_yaml}')
print(f'Classes ({len(class_names)}):', class_names)

def plot_image_with_boxes(image_path, label_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Failed to read image: {image_path}')
    h, w, _ = img.shape

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f.readlines():
                cls, x, y, bw, bh = map(float, line.strip().split())
                cls = int(cls)
                # YOLO normalized -> pixel coords
                x1 = int((x - bw / 2) * w)
                y1 = int((y - bh / 2) * h)
                x2 = int((x + bw / 2) * w)
                y2 = int((y + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    img,
                    class_names[cls],
                    (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2,
                )

    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

if not os.path.isdir(images_dir):
    raise FileNotFoundError(
        'Train images folder not found. Check dataset_location or data.yaml. '
        f'Got: {images_dir}'
    )
if not os.path.isdir(labels_dir):
    raise FileNotFoundError(
        'Train labels folder not found. Check dataset_location or data.yaml. '
        f'Got: {labels_dir}'
    )

# Show 3 sample images
sample_files = sorted(os.listdir(images_dir))[:3]
for file in sample_files:
    img_path = os.path.join(images_dir, file)
    lbl_path = os.path.join(labels_dir, file.rsplit('.', 1)[0] + '.txt')
    plot_image_with_boxes(img_path, lbl_path)

# Explanation
# - This is a sanity check and not part of training.
# - If labels do not match the objects visually, check data.yaml and paths.

#  Loading a Pre-trained YOLO26 Model

We start from **`yolo26s.pt`** (small = balanced speed/accuracy) pretrained on **COCO** (80 everyday classes).

Fine-tuning means: **reuse COCO's learned features**, then adjust the output head for our 11 self-driving classes. That is why we don't need millions of images — the model already knows what a "car" looks like from COCO.

Model size options:
- `yolo26n.pt` → Nano (≈5M params) — fastest, lightest
- `yolo26s.pt` → Small (≈19M params) — better accuracy, good balance ✅
- `yolo26m.pt` → Medium (≈42M params) — strongest for a class project
- `yolo26l.pt` / `yolo26x.pt` → Large / XLarge (≈51M / ≈113M params) — highest accuracy, slowest

In [ ]:
from pathlib import Path

model = YOLO(str(Path.cwd() / 'weights' / 'yolo' / 'yolo26s.pt'))
print('YOLO26s loaded with COCO pretrained weights!')

#  Fine-Tuning the Model (run if weights not yet exist)

Key training arguments:

| Arg | Meaning | Our value |
|---|---|---|
| `data` | Path to data.yaml | DGX path |
| `epochs` | Number of passes over the dataset | 30 |
| `imgsz` | Input image size (square) | 512 |
| `batch` | Images per training step | 16 |
| `patience` | Early stop if no improvement for N epochs | 10 |
| `device` | `cuda` = GPU, `cpu` = CPU | auto |

⏱️ **Expected time on a DGX GPU: 15-40 min (varies by GPU).**
Ultralytics prints a progress bar per epoch so you can see the ETA.

Results are auto-saved to `runs_output/detect/selfdriving_v1/` including:
- `weights/best.pt` -> best-performing weights inside the run folder
- `results.png` -> loss + mAP curves
- `confusion_matrix.png`

The later save/export cell copies the final `best.pt` and `results.png` into `weights/yolo/` for reuse in the other notebooks.

In [ ]:
#  Fine-Tuning the Model (run if weights not yet exist)
from pathlib import Path

project_dir = Path.cwd() / 'runs_output' / 'detect'
project_dir.mkdir(parents=True, exist_ok=True)

results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=512,
    batch=16,
    patience=10,
    project=str(project_dir),
    name='selfdriving_v1',
    device=device,
)

# Save the run directory for later cells (paths are auto-incremented)
save_dir = str(model.trainer.save_dir)
print(f"\nRun directory: {save_dir}")

# Explanation
# - Fine-tuning = predict -> loss -> backward -> optimizer.step (same loop as the ViT notebook).
# - Ultralytics hides the loop inside .train(), but it is the same idea.
# - After training, `model` automatically holds the best weights.

### Look at the Training Curves

`results.png` shows loss (should decrease) and mAP (should increase) over epochs.
If mAP flatlined early, more epochs won't help much.

In [ ]:
from pathlib import Path

output_root = Path.cwd() / 'runs_output' / 'detect'
candidate_run_dirs = []

if 'save_dir' in globals() and save_dir:
    candidate_run_dirs.append(Path(save_dir))

if output_root.exists():
    candidate_run_dirs.extend(
        sorted(
            [path for path in output_root.glob('selfdriving*') if path.is_dir()],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

results_png = None
for run_dir in candidate_run_dirs:
    candidate = run_dir / 'results.png'
    if candidate.exists():
        save_dir = str(run_dir)
        results_png = candidate
        break

if results_png is None:
    print('No training curves found. Run the fine-tuning cell first, or make sure runs_output/detect/selfdriving*/results.png exists.')
else:
    print(f'Using training run: {save_dir}')
    plt.figure(figsize=(14, 8))
    plt.imshow(Image.open(results_png))
    plt.axis('off')
    plt.show()

**Recall:** Of all real objects in the image, how many did the model find?
$$
\text{Recall} = \frac{TP}{TP + FN}
$$
High recall means fewer missed objects.

**Precision:** Of all predicted boxes, how many were correct?
$$
\text{Precision} = \frac{TP}{TP + FP}
$$
High precision means fewer false alarms.

**mAP (mean Average Precision):** A single score that summarizes precision-recall behavior.
For each class, sweep a confidence threshold, compute the precision-recall curve, then take the area under that curve (Average Precision).
mAP is the mean of those AP values across all classes.

**mAP@0.5:** Uses IoU threshold $0.5$ to decide if a box is correct. This is more forgiving.

**mAP@0.5:0.95:** Averages mAP over IoU thresholds from $0.5$ to $0.95$ in steps of $0.05$. This is stricter and usually lower.

# 📊 Part 1 — Formal Evaluation (the numbers for your report)

**Evaluation** = running the model on a held-out split and computing objective metrics.
We use `model.val()` on the **validation** split (Ultralytics uses the `val:` path from `data.yaml`).

**Key metrics:**
- **Precision** — of the boxes we predicted, how many were correct?
- **Recall** — of all real objects, how many did we find?
- **mAP@0.5** — mean Average Precision at IoU threshold 0.5 (main benchmark number).
- **mAP@0.5:0.95** — stricter; averaged across IoU thresholds 0.5 → 0.95.

A solid YOLO26s on this dataset typically reaches **mAP@0.5 ≈ 0.65–0.80**.

In [ ]:
from pathlib import Path

if 'dataset_location' not in globals() or not dataset_location:
    env_path = os.getenv('DATASET_DIR')
    if env_path and os.path.isdir(env_path):
        dataset_location = env_path
    else:
        candidates = [
            os.path.join(os.getcwd(), 'Self-Driving-Car-3'),
            os.path.join(os.getcwd(), 'self-driving-car'),
        ]
        dataset_location = next((path for path in candidates if os.path.isdir(path)), '')

if 'data_yaml' not in globals() or not data_yaml:
    if not dataset_location:
        raise RuntimeError(
            'Dataset path is missing. Run the dataset setup cell first, set DATASET_DIR, or place the dataset at ./Self-Driving-Car-3 or ./self-driving-car.'
        )
    data_yaml = os.path.join(dataset_location, 'data.yaml')

if not os.path.isfile(data_yaml):
    raise FileNotFoundError(f'data.yaml not found: {data_yaml}')

output_root = Path.cwd() / 'runs_output' / 'detect'
candidate_run_dirs = []

if 'save_dir' in globals() and save_dir:
    candidate_run_dirs.append(Path(save_dir))

if output_root.exists():
    candidate_run_dirs.extend(
        sorted(
            [path for path in output_root.glob('selfdriving*') if path.is_dir()],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

resolved_run_dir = None
for run_dir in candidate_run_dirs:
    best_weights = run_dir / 'weights' / 'best.pt'
    if best_weights.exists():
        resolved_run_dir = run_dir
        save_dir = str(run_dir)
        eval_model = YOLO(str(best_weights))
        break
else:
    canonical_best = Path.cwd() / 'weights' / 'yolo' / 'best.pt'
    if canonical_best.exists():
        eval_model = YOLO(str(canonical_best))
    else:
        # Refuse to fall back to the COCO-pretrained base model (yolo26s.pt):
        # evaluating an 80-class COCO head against the 11-class Self-Driving-Car-3
        # split would print meaningless P/R/mAP numbers that could end up in the report.
        raise RuntimeError(
            'No fine-tuned best.pt found. Train Stage 1 first or place '
            'weights/yolo/best.pt. Refusing to evaluate with COCO-pretrained '
            'yolo26s.pt because the metrics would not describe this project.'
        )

print(f'Using data.yaml: {data_yaml}')
val_results = eval_model.val(data=data_yaml)

print(f"\nPrecision     : {val_results.box.mp:.3f}")
print(f"Recall        : {val_results.box.mr:.3f}")
print(f"mAP@0.5       : {val_results.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {val_results.box.map:.3f}")

# Confusion matrix (which classes get confused with which?)
if resolved_run_dir is not None:
    cm_path = resolved_run_dir / 'confusion_matrix.png'
    if cm_path.exists():
        plt.figure(figsize=(10, 10))
        plt.imshow(Image.open(cm_path))
        plt.axis('off')
        plt.title('Confusion Matrix')
        plt.show()
    else:
        print('Confusion matrix not found in the resolved run directory.')
else:
    print('Using canonical or in-memory weights; no run-specific confusion matrix was resolved.')

# Explanation
# - val_results.box.* holds aggregated metrics over all classes.
# - For per-class metrics, check val_results.box.maps (one value per class).

# 🔍 Part 2 — Qualitative Demo on a Single Test Image

Numbers are nice, but it helps to **see** the model work. We pick a random image from the test split and plot the predicted boxes.

`conf=0.4` filters out low-confidence boxes to keep the output clean.

In [ ]:
from pathlib import Path

if 'dataset_location' not in globals() or not dataset_location:
    env_path = os.getenv('DATASET_DIR')
    if env_path and os.path.isdir(env_path):
        dataset_location = env_path
    else:
        candidates = [
            os.path.join(os.getcwd(), 'Self-Driving-Car-3'),
            os.path.join(os.getcwd(), 'self-driving-car'),
        ]
        dataset_location = next((path for path in candidates if os.path.isdir(path)), '')
    if not dataset_location:
        raise FileNotFoundError(
            'Dataset folder not found. Run the dataset setup cell first, set DATASET_DIR, or place the dataset at ./Self-Driving-Car-3 or ./self-driving-car.'
        )

test_dir = os.path.join(dataset_location, 'test', 'images')
if not os.path.isdir(test_dir):
    raise FileNotFoundError(f'Test images folder not found: {test_dir}')

test_files = sorted(os.listdir(test_dir))
if not test_files:
    raise FileNotFoundError(f'No test images found in {test_dir}')

test_img = os.path.join(test_dir, test_files[0])

output_root = Path.cwd() / 'runs_output' / 'detect'
candidate_run_dirs = []

if 'save_dir' in globals() and save_dir:
    candidate_run_dirs.append(Path(save_dir))

if output_root.exists():
    candidate_run_dirs.extend(
        sorted(
            [path for path in output_root.glob('selfdriving*') if path.is_dir()],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

for run_dir in candidate_run_dirs:
    best_weights = run_dir / 'weights' / 'best.pt'
    if best_weights.exists():
        demo_model = YOLO(str(best_weights))
        save_dir = str(run_dir)
        break
else:
    canonical_best = Path.cwd() / 'weights' / 'yolo' / 'best.pt'
    if canonical_best.exists():
        demo_model = YOLO(str(canonical_best))
    else:
        # Refuse to fall back to the COCO-pretrained base model (yolo26s.pt):
        # the demo is meant to showcase Self-Driving-Car-3 detections, and a COCO
        # head would draw boxes for unrelated classes, misleading the audience.
        raise RuntimeError(
            'No fine-tuned best.pt found. Train Stage 1 first or place '
            'weights/yolo/best.pt. Refusing to demo with COCO-pretrained '
            'yolo26s.pt because its classes are not the project classes.'
        )

results_test = demo_model(test_img, conf=0.4)

annotated = results_test[0].plot()
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('Fine-tuned YOLO26 - Dashcam Detections')
plt.axis('off')
plt.show()

### Exploring the `results` Object

Ultralytics wraps predictions in a `Results` object. The most useful attributes:

| Attribute | What it contains |
|---|---|
| `.boxes.xyxy` | Box corners in pixels `[x1, y1, x2, y2]` |
| `.boxes.conf` | Confidence score (0–1) per box |
| `.boxes.cls` | Class index per box |
| `.names` | Dict mapping class index → class name |

In [ ]:
if 'results_test' not in globals() or not results_test:
    raise RuntimeError('Missing test-image predictions. Run the previous qualitative demo cell first.')

result = results_test[0]

boxes = result.boxes.xyxy.cpu().numpy()
confs = result.boxes.conf.cpu().numpy()
clses = result.boxes.cls.cpu().numpy().astype(int)
names = result.names

print(f'Found {len(boxes)} objects:\n')
for box, conf, cls in zip(boxes, confs, clses):
    print(f'  {names[cls]:15s}  conf={conf:.2f}  box={box.astype(int).tolist()}')

# 🎥 Part 3 — Video Demo (DGX Path Input)

Place a dashcam clip on the DGX (scp it from your Mac if needed).
The code below processes every frame and **saves an annotated output video** on the DGX.

In [ ]:
import glob
import os
from pathlib import Path

from ultralytics import YOLO

project_root = Path.cwd()
output_root = project_root / 'runs_output' / 'detect'
output_root.mkdir(parents=True, exist_ok=True)

# Set a DGX path to a local video file
video_path = project_root / 'original_videos' / 'dashcam.mp4'
assert video_path.exists(), 'Video path not found on DGX'

# Prefer the newly trained run weights, then the exported canonical weights.
# Refuse to silently fall back to the COCO-pretrained base model: this notebook
# is about the fine-tuned Self-Driving-Car-3 detector, and using base weights
# would silently degrade Stage 1 output and propagate to Stages 2b / 3.
run_best = Path(globals().get('save_dir', '')) / 'weights' / 'best.pt' if globals().get('save_dir') else None
canonical_best = project_root / 'weights' / 'yolo' / 'best.pt'
if run_best and run_best.exists():
    predict_model = YOLO(str(run_best))
    print(f'Using fine-tuned weights from this session: {run_best}')
elif canonical_best.exists():
    predict_model = YOLO(str(canonical_best))
    print(f'Using exported fine-tuned weights: {canonical_best}')
else:
    raise RuntimeError(
        'No fine-tuned YOLO weights found. Looked for '
        f'{run_best} (this session\'s training run) and '
        f'{canonical_best} (canonical export). '
        'Run the fine-tuning cell first, or place a trained best.pt at weights/yolo/best.pt.'
    )

# Run detection on every frame - save under runs_output/detect/predict*/
predict_model(str(video_path), save=True, conf=0.4, device=device, project=str(output_root))

# Find the annotated output and print its path
out_candidates = sorted(
    glob.glob(str(output_root / 'predict*' / '*.avi')) + glob.glob(str(output_root / 'predict*' / '*.mp4')),
    key=os.path.getmtime,
)
if out_candidates:
    out_video = out_candidates[-1]
    print(f'Output file: {out_video}')
else:
    print('No output video found - check runs_output/detect/ manually.')

# Explanation
# - Ultralytics writes .avi by default; recent versions may use .mp4.
# - Use scp to copy the output video back to your Mac if needed.

**Note (MP4 export):** The next cell uses `ffmpeg` to convert the newest output under `runs_output/detect/predict*/` into `runs_output/detect/predict*/converted_mp4/`. If `ffmpeg` is missing, install it with `sudo apt-get install ffmpeg`.


In [ ]:
# Convert the latest prediction video to MP4 (~100MB target)
import glob
import os
import shutil
import subprocess
from pathlib import Path

target_size_mb = 100
output_root = Path.cwd() / 'runs_output' / 'detect'

if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    print('ffmpeg/ffprobe not found. Install with: sudo apt-get install ffmpeg')
else:
    predict_dirs = glob.glob(str(output_root / 'predict*'))
    if not predict_dirs:
        raise FileNotFoundError('No runs_output/detect/predict* folders found. Run the video inference cell first.')
    latest_predict = max(predict_dirs, key=os.path.getmtime)
    video_candidates = glob.glob(os.path.join(latest_predict, '*.avi')) + glob.glob(
        os.path.join(latest_predict, '*.mp4')
    )
    if not video_candidates:
        raise FileNotFoundError(f'No video files found in {latest_predict}')
    input_video = max(video_candidates, key=os.path.getmtime)
    out_dir = os.path.join(latest_predict, 'converted_mp4')
    os.makedirs(out_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(input_video))[0]
    output_video = os.path.join(out_dir, f'{base_name}_compressed.mp4')

    probe_cmd = [
        'ffprobe',
        '-v',
        'error',
        '-show_entries',
        'format=duration',
        '-of',
        'csv=p=0',
        input_video,
    ]
    duration = float(subprocess.check_output(probe_cmd).decode().strip())
    if duration <= 0:
        raise ValueError('Invalid video duration from ffprobe')

    target_bits = target_size_mb * 1024 * 1024 * 8
    bitrate_kbps = int((target_bits / duration) / 1000)
    bitrate_kbps = max(500, min(bitrate_kbps, 12000))
    buf_kbps = bitrate_kbps * 2

    ffmpeg_cmd = [
        'ffmpeg',
        '-y',
        '-i',
        input_video,
        '-c:v',
        'libx264',
        '-preset',
        'medium',
        '-b:v',
        f'{bitrate_kbps}k',
        '-maxrate',
        f'{bitrate_kbps}k',
        '-bufsize',
        f'{buf_kbps}k',
        '-c:a',
        'aac',
        '-b:a',
        '128k',
        '-movflags',
        '+faststart',
        output_video,
    ]
    subprocess.run(ffmpeg_cmd, check=True)
    size_mb = os.path.getsize(output_video) / (1024 * 1024)
    print(f'Input:  {input_video}')
    print(f'Output: {output_video}')
    print(f'Output size: {size_mb:.1f} MB')

# 💾 Save the Fine-Tuned Weights (run if not yet saved )

Store `best.pt` in `weights/yolo/`

In [ ]:
from pathlib import Path
import shutil

output_root = Path.cwd() / 'runs_output' / 'detect'
candidate_run_dirs = []

if 'save_dir' in globals() and save_dir:
    candidate_run_dirs.append(Path(save_dir))

if output_root.exists():
    candidate_run_dirs.extend(
        sorted(
            [path for path in output_root.glob('selfdriving*') if path.is_dir()],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

resolved_run_dir = None
for run_dir in candidate_run_dirs:
    best_candidate = run_dir / 'weights' / 'best.pt'
    if best_candidate.exists():
        resolved_run_dir = run_dir
        save_dir = str(run_dir)
        break

if resolved_run_dir is None:
    raise FileNotFoundError(
        'No trained run with best.pt was found. Run the fine-tuning cell first, or make sure runs_output/detect/selfdriving*/weights/best.pt exists.'
    )

best_pt = resolved_run_dir / 'weights' / 'best.pt'
results_png = resolved_run_dir / 'results.png'
target_dir = Path('/home/dongyuan/Desktop/computer_vision/weights/yolo')
target_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_pt, target_dir / 'best.pt')
if results_png.exists():
    shutil.copy2(results_png, target_dir / 'results.png')

print(f'Using training run: {resolved_run_dir}')
print(f'Saved best.pt and results.png to: {target_dir}')

# To copy to your Mac, use scp from a terminal, e.g.:
# scp dongyuan@<dgx-host>:/home/dongyuan/Desktop/computer_vision/weights/yolo/best.pt ./

# 🧠 Interpreting Results — What to Report

When presenting this project, cover:

1. **Dataset**: what it contains, how many images, how many classes.
2. **Model choice**: why YOLOv10n (speed vs accuracy trade-off).
3. **Training curves**: show `results.png` — did loss decrease? did mAP plateau?
4. **Metrics**: precision, recall, mAP@0.5. Note which classes were weakest (check confusion matrix).
5. **Qualitative results**: the annotated dashcam video and/or live webcam demo — the "wow" moment.
6. **Limitations**: small objects (far-away cars), night scenes, weather, class imbalance.
7. **Next steps**: more epochs, bigger model (`yolov10s`), stronger augmentation, add tracking.

# 💡 Student Tasks

1. **Train longer** — bump `epochs` to 50 with `patience=15`. Does mAP improve?
2. **Bigger model** — try `yolov10s.pt` or `yolo11n.pt` and compare training time vs mAP gain.
3. **Different dataset** — try **Tsinghua-Tencent TT100K** (traffic signs only) and compare.
4. **Add tracking** — replace `model(video_path, save=True)` with `model.track(video_path, tracker="bytetrack.yaml", save=True)` so each vehicle keeps a persistent ID across frames.
5. **Measure FPS** — time the video inference and compute frames/second. Is it real-time (>30 FPS)?
6. **Per-class analysis** — print `val_results.box.maps` and discuss which classes are hardest.